Bayesian analysis of IMDB 50,000 movie reviews dataset
The conditional probability chosen is P(Negative | keyword)

In [1]:
import csv
import re

DATA_PATH = "IMDB Dataset.csv"

In [2]:
def load_data(path):
    """Read the IMDB CSV and return a list of (review_text, sentiment) tuples."""
    data = []
    with open(path, encoding="utf-8", newline="") as f:
        reader = csv.reader(f)
        next(reader)                          # skip the header row
        for row in reader:
            data.append((row[0], row[1]))     # (review_text, sentiment)
    return data


In [3]:
def tokenize(text):
    """Lowercase the text, strip <br /> tags, and return the set of unique words."""
    text = text.lower()                       # 1. lowercase first
    text = text.replace("<br />", " ")        # 2. remove <br /> tags (space, not empty)
    words = re.findall(r"[a-z']+", text)      # 3. grab runs of letters/apostrophes
    words = [w.strip("'") for w in words]     # 4. remove quote marks from the ends
    return set(w for w in words if w)

In [4]:
def prepare(data):
    """Tokenize every review once and return a list of (set_of_words, sentiment) tuples."""
    return [(tokenize(text), sentiment) for text, sentiment in data]


In [5]:
def compute_posterior(prepared, keyword):
    """Return (prior, likelihood, marginal, posterior) for P(Negative | keyword)."""
    total = len(prepared)
    negative = 0          # reviews labelled "negative"
    with_keyword = 0      # reviews containing the keyword
    both = 0              # negative AND containing the keyword

    for words, sentiment in prepared:
        is_neg = sentiment == "negative"
        has_kw = keyword in words
        if is_neg:
            negative += 1
        if has_kw:
            with_keyword += 1
        if is_neg and has_kw:
            both += 1

    prior = negative / total
    likelihood = both / negative if negative > 0 else 0
    marginal = with_keyword / total if total > 0 else 0
    posterior = likelihood * prior / marginal if marginal > 0 else None

    if posterior is not None:
        assert abs(posterior - both / with_keyword) < 1e-9

    return prior, likelihood, marginal, posterior, with_keyword

In [6]:
prepared = prepare(load_data(DATA_PATH))
len(prepared)

50000

In [7]:
def print_results_table(prepared, keyword_groups):
    """Print one row per keyword for P(Negative | keyword)."""
    print(f"{'keyword':<12}{'type':<10}{'P(Neg)':>9}{'P(kw|Neg)':>11}{'P(kw)':>9}{'P(Neg|kw)':>11}{'reviews':>9}")
    for label, words in keyword_groups:
        for kw in words:
            prior, likelihood, marginal, posterior, with_keyword = compute_posterior(prepared, kw)
            if posterior is None:
                print(f"{kw:<12}{label:<10}never appears in the dataset")
                continue
            print(f"{kw:<12}{label:<10}{prior:>9.4f}{likelihood:>11.4f}{marginal:>9.4f}{posterior:>11.4f}{with_keyword:>9}")

In [8]:
POSITIVE_KEYWORDS = ["great", "excellent", "wonderful", "perfect"]
NEGATIVE_KEYWORDS = ["worst", "awful", "terrible", "waste"]

print_results_table(prepared, [("positive", POSITIVE_KEYWORDS), ("negative", NEGATIVE_KEYWORDS)])

keyword     type         P(Neg)  P(kw|Neg)    P(kw)  P(Neg|kw)  reviews
great       positive     0.5000     0.1640   0.2515     0.3259    12576
excellent   positive     0.5000     0.0274   0.0710     0.1926     3552
wonderful   positive     0.5000     0.0209   0.0556     0.1879     2778
perfect     positive     0.5000     0.0250   0.0533     0.2343     2667
worst       negative     0.5000     0.1609   0.0887     0.9073     4434
awful       negative     0.5000     0.1040   0.0576     0.9018     2882
terrible    negative     0.5000     0.0927   0.0540     0.8585     2699
waste       negative     0.5000     0.0944   0.0507     0.9313     2533


## Choice of conditional probability

We computed P(Negative | keyword) for each keyword, using Bayes' Theorem:

$$P(\text{Negative} \mid \text{keyword}) = \frac{P(\text{keyword} \mid \text{Negative})\, P(\text{Negative})}{P(\text{keyword})}$$

## How to read the results

- Negative keywords (worst, awful, terrible and waste) have a **high** posterior (0.8585 to 0.9313), so a review containing one of them is very likely to be negative.
- Positive keywords (great, excellent, wonderful and perfect) have a **low** posterior (0.1879 to 0.3259). This is expected: these words point toward positive reviews.

## Justification of keywords

We chose words that appear in enough reviews to give reliable estimates (the rarest, "waste", appears in 2,533 reviews) and that people use to express a clear feeling about a film.

## Findings

The negative keywords are the strongest evidence. Their P(Negative | keyword) ranges from 0.8585 (terrible) to 0.9313 (waste), so about 93% of reviews containing "waste" are negative. Among the positive keywords, "wonderful" (0.1879) and "excellent" (0.1926) are the strongest: about 81% of reviews containing them are positive.

The weakest keyword is "great" (0.3259), the closest to 0.5. It appears in 16.4% of negative reviews, but also in about 33.9% of positive reviews, which pulls its posterior toward 0.5. This makes it the least informative word.

The likelihoods are small. Even "worst" appears in only about 16% of negative reviews, so 84% of negative reviews do not contain it. One keyword cannot classify a review on its own, which is why real spam filters combine many words.

## Why the prior matters

Take a disease test that catches 90% of true cases and falsely flags 5% of healthy people. Test 1,000 people:

| Prevalence | Have it | True positives | Don't have it | False positives | P(has it \| positive test) |
|---|---|---|---|---|---|
| 10% | 100 | 90 | 900 | 45 | 90 / 135 ≈ 66.7% |
| 1% | 10 | 9 | 990 | about 50 (49.5) | 9 / 58.5 ≈ 15.4% |

The test is identical in both rows. Only the prior changed, and the posterior fell from about 67% to about 15%. At low prevalence, the many healthy people produce more false alarms than the few sick people produce true hits.

In our data the prior is exactly 0.5, because the dataset is split 25,000 / 25,000. If it were 90% positive reviews, P(Negative) would be 0.10. With the same likelihoods, the posterior for "worst" would drop from 0.9073 to about 0.52. The word would be just as informative, but negative reviews would be so rare that many reviews containing "worst" would be positive.

## Limitation

"great" had the weakest result (0.3259), likely because the method ignores context such as negation ("not great") and sarcasm.

## Assumptions

- A review counts once no matter how many times the word appears (we check presence, not frequency).
- Words are matched whole, so "awfully" does not count as "awful".
- Each keyword is analysed alone, ignoring all other words in the review.